# Simbolo aprendido con PDNO-PINN

Se aprende un simbolo $a_\theta:\mathbb{Z}^n\to\mathbb{C}$ con una MLP **sin restricciones estructurales**
(capas ocultas `Tanh` + `Dropout`, salida lineal para $\mathrm{Re}\,a_\theta$ e $\mathrm{Im}\,a_\theta$).

La perdida combina:

- **Trayectoria:** $\mathcal{L}_{\text{traj}} = \mathbb{E}\big[|e^{-t a_\theta(\mathbf{k})}\hat u_0(\mathbf{k}) - \hat u_{\text{ref}}(\mathbf{k},t)|^2\big]$
- **Parabolicidad:** $\mathcal{L}_{\text{par}} = \mathbb{E}\big[\max\big(0,\ |(a_\theta(\mathbf{k})+\lambda)^{-1}| - \kappa(1+|(\mathbf{k},\lambda)|^2)^{-m/2}\big)^2\big]$,
  sobre $\mathrm{Re}(\lambda)\ge 0$, $|(\mathbf{k},\lambda)|\ge\omega$.

$$\mathcal{L} = \lambda_{\text{traj}}\,\mathcal{L}_{\text{traj}} + \lambda_{\text{par}}\,\mathcal{L}_{\text{par}}$$

Como referencia se usa el calor, $a(\mathbf{k}) = |\mathbf{k}|^2$ en $\mathbb{T}^2$.

## Protocolo experimental

- **Datos:** $10^5$ muestras, particionadas en **entrenamiento (80%) y validacion (20%)**. No hay conjunto de prueba.
- **$\kappa$ como hiperparametro:** se barre una malla de valores y se conserva el **mas pequeno que el
  entrenamiento logra satisfacer** en toda la rejilla de comprobacion. $\kappa$ mas chico = cota mas exigente.
- **Arquitectura:** barrido de $\{2,3,4\}$ capas ocultas $\times$ $\{32,64,128\}$ neuronas.
- **Regularizacion:** dropout $0.2$ tras cada activacion oculta, y **early stopping** sobre la
  perdida total de validacion (se restauran los pesos de la mejor epoca).

In [ ]:
import sys
sys.path.insert(0, "../src")  # ruta para las clases

import numpy as np
import matplotlib.pyplot as plt
import torch

from parabolic_symbol.ParabolicityVerifier import ParabolicityVerifier
from parabolic_symbol.symbols.heat import HeatSymbol
from parabolic_symbol.symbols.mlp_symbol import MLPSymbol, default_device
from parabolic_symbol.training import (
    TrainingConfig,
    best_run,
    k_grid,
    make_trajectory_data,
    phase_grid,
    run_sweep,
    split_trajectory_data,
    sweep_table,
)

np.set_printoptions(precision=4, suppress=True)

device = default_device()
# En GPU se usa float32: la doble precision corre a 1/32-1/64 del rendimiento en
# tarjetas de consumo y aqui no aporta (ver comparacion float32/float64 en el repo).
dtype = torch.float32 if device.type == "cuda" else torch.float64
print(f"device = {device}  |  dtype = {dtype}")

## 1. Datos de referencia

$\hat u_{\text{ref}}(\mathbf{k},t) = e^{-t|\mathbf{k}|^2}\hat u_0(\mathbf{k})$, con $|k_i|\le k_{\max}$.
El tiempo se muestrea log-uniforme para que haya senal tanto en $|\mathbf{k}|$ pequeno como grande.

In [ ]:
n = 2
k_max = 8
reference = HeatSymbol(n=n)

data = make_trajectory_data(
    reference, k_max=k_max, n_samples=100_000, t_min=1e-3, t_max=1.0, seed=0
)
train_data, val_data = split_trajectory_data(data, val_fraction=0.2, seed=0)

print(f"entrenamiento: {train_data['k'].shape[0]:,} muestras")
print(f"validacion:    {val_data['k'].shape[0]:,} muestras")

## 2. Malla de $\kappa$

Se estima $\hat\kappa$ del simbolo de referencia con el verificador y se construye una malla alrededor.
Valores por debajo de $\hat\kappa$ son cotas mas exigentes que las que cumple el propio calor sobre esta rejilla.

In [ ]:
omega = 1.0
k_values = list(k_grid(n, k_max))
lam_values = [complex(re, im) for re in (0, 1, 5, 10, 25, 50) for im in (-50, -10, -2, 0, 2, 10, 50)]

kappa_ref = ParabolicityVerifier(reference).estimate_kappa(k_values, lam_values, omega=omega)
kappas = [round(factor * kappa_ref, 3) for factor in (0.75, 1.0, 1.25, 1.5, 2.0)]

print(f"kappa_hat referencia = {kappa_ref:.4f}")
print(f"malla de kappa       = {kappas}")

## 3. Barrido de hiperparametros

$5$ valores de $\kappa$ $\times$ $3$ profundidades $\times$ $3$ anchos $= 45$ entrenamientos.

Cada corrida se marca como **cumple** si el peor ratio
$|(a_\theta+\lambda)^{-1}| \big/ \kappa(1+|(\mathbf{k},\lambda)|^2)^{-m/2}$
sobre la rejilla fija de comprobacion es $\le 1$. Esa rejilla es independiente de los puntos
muestreados en la perdida, asi que la decision no esta contaminada por el entrenamiento.

In [ ]:
check_phase = phase_grid(n, k_max, lam_values, device=device, dtype=dtype)

base_config = TrainingConfig(
    epochs=300,
    batch_size=1024,
    lr=1e-3,
    lr_final=1e-5,
    omega=omega,
    lambda_trajectory=1.0,
    lambda_parabolicity=1.0,
    k_max=k_max,
    lam_min=1e-2,
    lam_max=1e2,
    grad_clip=1.0,
    patience=30,        # early stopping
    n_phase_eval=8192,  # puntos (k, lambda) fijos para las metricas por epoca
    log_every=0,        # el barrido ya imprime una linea por corrida
    seed=0,
)

runs = run_sweep(
    train_data,
    val_data,
    n=n,
    m=2.0,
    kappas=kappas,
    depths=[2, 3, 4],
    widths=[32, 64, 128],
    base_config=base_config,
    check_phase=check_phase,
    dropout=0.2,
    input_scale=k_max,
    output_scale=k_max**2,
    device=device,
    dtype=dtype,
)

## 4. Resultados del barrido

In [ ]:
table = sweep_table(runs)
table.sort_values(["kappa", "val_rel_l2"]).reset_index(drop=True)

In [ ]:
resumen = table.groupby("kappa").agg(
    cumplen=("cumple", "sum"),
    corridas=("cumple", "count"),
    mejor_max_ratio=("max_ratio", "min"),
    mejor_val_rel_l2=("val_rel_l2", "min"),
)
resumen

### Seleccion: el $\kappa$ mas chico que se logra satisfacer

Entre las corridas que cumplen se toma el $\kappa$ minimo; a igualdad de $\kappa$, la de mejor
error relativo de validacion.

In [ ]:
best = best_run(runs)
if best is None:
    raise RuntimeError("ningun kappa de la malla se logro satisfacer; ampliar la malla hacia arriba")

symbol = best.symbol
print(f"kappa minimo satisfecho = {best.kappa:.4f}   (kappa_hat referencia = {kappa_ref:.4f})")
print(f"arquitectura elegida    = {best.n_layers} capas x {best.width} neuronas, dropout 0.2")
print(f"max ratio en la rejilla = {best.max_ratio:.4f}  (<= 1 => la cota se cumple)")
print(f"val rel_l2              = {best.val_rel_l2:.4f}")
print(f"early stopping          = mejor epoca {best.best_epoch} de {best.epochs_run} corridas")

## 5. Curvas de entrenamiento

Las metricas de ambos conjuntos se calculan al final de cada epoca **en modo eval** (dropout apagado),
asi las dos curvas son directamente comparables. La linea punteada marca la epoca cuyos pesos
restauro el early stopping.

In [ ]:
history = best.result.history
epochs = np.arange(1, len(history["train"]["total"]) + 1)
floor = 1e-12  # recorte para la escala logaritmica

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].semilogy(epochs, history["train"]["total"], label="entrenamiento")
axes[0].semilogy(epochs, history["val"]["total"], label="validacion")
axes[0].axvline(best.best_epoch, color="k", ls=":", lw=1, label="mejor epoca")
axes[0].set_xlabel("epoca")
axes[0].set_ylabel("perdida total")
axes[0].set_title("Perdida vs epoca")
axes[0].legend()
axes[0].grid(alpha=0.3)

for split, style in (("train", "-"), ("val", "--")):
    axes[1].semilogy(epochs, np.maximum(history[split]["trajectory"], floor), style,
                     color="C0", label=f"trayectoria ({split})")
    axes[1].semilogy(epochs, np.maximum(history[split]["parabolicity"], floor), style,
                     color="C3", label=f"parabolicidad ({split})")
axes[1].axvline(best.best_epoch, color="k", ls=":", lw=1)
axes[1].set_xlabel("epoca")
axes[1].set_ylabel("componente de la perdida")
axes[1].set_title("Componentes de la perdida vs epoca")
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(epochs, history["train"]["rel_l2"], label="entrenamiento")
axes[0].plot(epochs, history["val"]["rel_l2"], label="validacion")
axes[0].axvline(best.best_epoch, color="k", ls=":", lw=1, label="mejor epoca")
axes[0].set_xlabel("epoca")
axes[0].set_ylabel(r"error relativo $L^2$ de la trayectoria")
axes[0].set_title("Metrica vs epoca")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(epochs, history["train"]["par_max_ratio"], label="entrenamiento")
axes[1].plot(epochs, history["val"]["par_max_ratio"], label="validacion")
axes[1].axhline(1.0, color="k", ls="--", lw=1, label="cota (ratio = 1)")
axes[1].axvline(best.best_epoch, color="k", ls=":", lw=1)
axes[1].set_xlabel("epoca")
axes[1].set_ylabel("peor ratio de parabolicidad")
axes[1].set_title("Violacion de parabolicidad vs epoca")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

La perdida de parabolicidad vale exactamente $0$ en las epocas donde ningun punto muestreado viola la cota
(en la grafica se recorta a $10^{-12}$ para la escala logaritmica).

## 6. Simbolo aprendido vs. referencia

In [ ]:
k_arr = np.array(k_values)
a_true = np.array([reference(k)[0, 0] for k in k_values])
a_learn = symbol.evaluate_batch(k_arr)   # un solo forward por lotes
norm_k2 = np.sum(k_arr**2, axis=1)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(norm_k2, a_learn.real, s=10, label=r"Re $a_\theta$")
axes[0].scatter(norm_k2, a_learn.imag, s=10, label=r"Im $a_\theta$")
axes[0].plot(np.sort(norm_k2), np.sort(norm_k2), "k--", lw=1, label=r"$|k|^2$")
axes[0].set_xlabel(r"$|k|^2$")
axes[0].legend()
axes[0].grid(alpha=0.3)

im = axes[1].imshow(
    np.abs(a_learn - a_true).reshape(2 * k_max + 1, 2 * k_max + 1),
    extent=[-k_max - 0.5, k_max + 0.5, k_max + 0.5, -k_max - 0.5],
)
axes[1].set_title(r"$|a_\theta(k) - |k|^2|$")
axes[1].set_xlabel(r"$k_2$")
axes[1].set_ylabel(r"$k_1$")
fig.colorbar(im, ax=axes[1])
plt.tight_layout()
plt.show()

rel = np.abs(a_learn - a_true) / np.maximum(norm_k2, 1)
print(f"error relativo medio = {rel.mean():.3f}, maximo = {rel.max():.3f}")

Para $|\mathbf{k}|$ grande $e^{-t|\mathbf{k}|^2}\approx 0$ en casi todo el rango de $t$, asi que la perdida de
trayectoria da poca informacion alli y el error es mayor.

## 7. Verificacion con `ParabolicityVerifier`

`MLPSymbol` hereda de `Symbol`, asi que se pasa al verificador sin cambios.

In [ ]:
verifier = ParabolicityVerifier(symbol)

kappa_learned = verifier.estimate_kappa(k_values, lam_values, omega=omega)
results = verifier.check_region(k_values, lam_values, omega=omega, kappa=best.kappa)

print(f"{symbol!r}")
print(f"kappa_hat del simbolo aprendido = {kappa_learned:.4f}  (kappa usada = {best.kappa:.4f})")
print(f"Se cumple en toda la rejilla ({len(results)} puntos): {all(r.holds for r in results)}")
print(f"max ratio = {max(r.ratio for r in results):.4f}")

In [ ]:
worst = sorted(results, key=lambda r: -r.ratio)[:5]
for r in worst:
    print(f"k={r.k}, lambda={r.lam}, ratio={r.ratio:.4f}")

## 8. Fuera de la ventana de entrenamiento

La salida de la red (Tanh + capa lineal) esta acotada, asi que $a_\theta$ no puede crecer como $|\mathbf{k}|^m$
para todo $\mathbf{k}\in\mathbb{Z}^n$. La parabolicidad solo puede garantizarse en la ventana muestreada;
fuera de ella se espera que la cota falle.

In [ ]:
# Ratio de parabolicidad sobre k_np x lam_values, con un solo forward por lotes.
def ratios_on_grid(symbol, k_np, lam_values, omega, kappa):
    kk = np.repeat(k_np, len(lam_values), axis=0)
    ll = np.tile(np.asarray(lam_values, dtype=complex), len(k_np))
    a = symbol.evaluate_batch(kk)
    norm = np.sqrt(np.sum(kk**2, axis=1) + np.abs(ll) ** (2.0 / symbol.m))
    keep = (ll.real >= 0) & (norm >= omega)
    kk, ll, norm, a = kk[keep], ll[keep], norm[keep], a[keep]
    bound = kappa * (1.0 + norm**2) ** (-symbol.m / 2.0)
    return kk, (1.0 / np.abs(a + ll)) / bound


k_far = k_grid(n, 3 * k_max)
kk_far, ratio_far = ratios_on_grid(symbol, k_far, lam_values, omega, best.kappa)
fails = ratio_far > 1.0

print(f"|k_i| <= {3 * k_max}: se cumple en todos los puntos: {not fails.any()}")
print(f"puntos que fallan: {fails.sum()} de {ratio_far.size}")
if fails.any():
    print(f"menor max|k_i| con falla: {int(np.abs(kk_far[fails]).max(axis=1).min())}")